optimising this version for NOTS

#### 1. Part 1 builds the 2D 2-particle correlation in both the standard and WTA frames. Clusters the jets according to the two definitions, transforms things to the jet frame, makes the signal distribution, the background distribution, and then the final normalised yield.

BEFORE RUNNING:
1. check input (f = cell) and output (def saveOutput) locations
2. check multiplicity bins (in final analysis cell)


In [17]:
import ROOT
import math
import fastjet
import os
import ctypes
import gc
import vector
import awkward
import uproot


In [4]:
# 1. Force ROOT into headless batch mode - i.e. turn off pop up graphics
ROOT.gROOT.SetBatch(True)

In [5]:
#from preliminary file

# INPUT DATA IS OPENED HERE

#f = ROOT.TFile.Open("/Users/rohanjagadeesan/Desktop/Code/li_lab/WinnerTakeAll/data/parton_data.root")
#f = ROOT.TFile.Open("/Users/rohanjagadeesan/Desktop/Code/li_lab/WinnerTakeAll/data/pp_parton_cascade_batch0_0.root")

f = ROOT.TFile.Open("/storage/hpc/work/wl33/ampt_xiao/3mb/nch60_pt500")

tree = f.Get("trackTree") #opening up the tracktree

jet_radius = 0.8 #max radius accepted by fastjet is 1000. radius value in prl paper is 0.8 in lab frame.

wta_def = fastjet.JetDefinition(fastjet.antikt_algorithm, jet_radius, fastjet.WTA_pt_scheme) #winner take all definition
std_def = fastjet.JetDefinition(fastjet.antikt_algorithm, jet_radius) #standard E scheme definition

OSError: Failed to open file /storage/hpc/work/wl33/ampt_xiao/3mb/nch60_pt500

Error in <TFile::TFile>: file /storage/hpc/work/wl33/ampt_xiao/3mb/nch60_pt500 does not exist


new additions

In [6]:
#from gemini - cross check
# optimising looping through the ttree
def optimize_tree_io(tree):
    # Disable every branch to prevent loading unused data into RAM
    tree.SetBranchStatus("*", 0) # "*" means all branches, 0 means not processing 
    
    # Explicitly enable ONLY the branches your analysis reads
    active_branches = [
        "genJetPt", "genJetEta", 
        "genDau_pt", "genDau_eta", "genDau_phi", "genDau_chg"
    ]

    for branch in active_branches:
        tree.SetBranchStatus(branch, 1) # 1 means being processed, so these branches get switched on

In [9]:
#storing jet data in a dictionary instead of looping through the whole tree multiple times


# ROOT histogram binning parameters     (i think this is better kept outside the function? maybe not though.)
eta_bins, eta_min, eta_max = 41, -6.15, 6.15
phi_bins = 33
phi_min = -(math.pi/2.0) + (math.pi/32.0)
phi_max = (3*math.pi/2.0) + (math.pi/32.0)

def initialize_all_bins(analysis_bins):
    '''
    initialising 
        1. the dictionary for jet info
        2. as well as initialising the histograms
    input: 
        - analysis bins
    mutated:
        - histograms with appropriate bin name are initialised but not returned
    output:
        - bin_data, the dictionary of jet info. Keys are multiplicity bins, and each key has items:
            - wta_stats: number of signal pairs, number of jets, and total Nch for this bin
            - std_stats: same but std
            - wta_hists: signal, background, and EPD hists
            - std_hists: same but std
        
    '''

    bin_data = {} #dict of dicts. The keys are multiplicity bins
    
    
    for mult_bin in analysis_bins:
        # Create a unique string identifier for the bin
        b_name = f"{mult_bin[0]}_{mult_bin[1]}" #not keeping the if statement that checks whether bin is one or two numbers because it should be 2 numbers.
            
        # Initialize dictionaries for this specific bin
        bin_data[b_name] = {
            'wta_stats': {'num_jets': 0, 'num_pairs': 0, 'Nch': 0},
            'std_stats': {'num_jets': 0, 'num_pairs': 0, 'Nch': 0},
            'wta_hists': {},
            'std_hists': {}
        }
        
        # Initialize WTA Histograms with SetDirectory(0)
        hSig_wta = ROOT.TH2D(f"hSig_WTA_{b_name}", ";#Delta#eta*;#Delta#phi*", eta_bins, eta_min, eta_max, phi_bins, phi_min, phi_max)
        hBkg_wta = ROOT.TH2D(f"hBkg_WTA_{b_name}", ";#Delta#eta*;#Delta#phi*", eta_bins, eta_min, eta_max, phi_bins, phi_min, phi_max)
        hEPD_wta = ROOT.TH2D(f"hEPD_WTA_{b_name}", ";#eta*;#phi*", 150, 0, 10, 120, -4, 4)
        
        for hist in [hSig_wta, hBkg_wta, hEPD_wta]:
            hist.SetDirectory(0)    #detaches the histograms from root gDirectory
            
        bin_data[b_name]['wta_hists'] = {'sig': hSig_wta, 'bkg': hBkg_wta, 'epd': hEPD_wta}
        
        # Initialize STD Histograms with SetDirectory(0)
        hSig_std = ROOT.TH2D(f"hSig_STD_{b_name}", ";#Delta#eta*;#Delta#phi*", eta_bins, eta_min, eta_max, phi_bins, phi_min, phi_max)
        hBkg_std = ROOT.TH2D(f"hBkg_STD_{b_name}", ";#Delta#eta*;#Delta#phi*", eta_bins, eta_min, eta_max, phi_bins, phi_min, phi_max)
        hEPD_std = ROOT.TH2D(f"hEPD_STD_{b_name}", ";#eta*;#phi*", 150, 0, 10, 120, -4, 4)
        
        for hist in [hSig_std, hBkg_std, hEPD_std]:
            hist.SetDirectory(0)
            
        bin_data[b_name]['std_hists'] = {'sig': hSig_std, 'bkg': hBkg_std, 'epd': hEPD_std}
        
    return bin_data




In [8]:
# helper function to put jet data into the dictionary

def get_bin_key(jet_mult, analysis_bins):
    '''
    outputs the multiplicity bin key for a given multiplicity
    
    inputs:
        - jet multiplicity
        - the analysis bins
    output:
        - the bin_data key for this multiplicity 
    '''

    for mult_bin in analysis_bins:
        if jet_mult >= mult_bin[0] and jet_mult < mult_bin[1]:
            return f"{mult_bin[0]}_{mult_bin[1]}"
    
    return None # Jet falls outside all defined bins

In [16]:
#running the analysis - one loop of the ttreee instead of many now.

def runAnalysis_SinglePass(tree, analysis_bins):
    '''
    runs the whole analysis. 
    inputs:
        - tree, the tracktree from the relevant root file
        - analysis_bins, the multiplicity bins we are using for the analysis
    '''
    
    # 1. Optimize IO footprint
    optimize_tree_io(tree) #makes sure only relevant branches are active for processing
    
    # 2. Setup master dictionary mapping all bins
    bin_data = initialize_all_bins(analysis_bins)
    
    # 3. The Single Pass Loop
    for event in tree:
        for ijet in range(event.genJetPt.size()): #ijet is an integer index for each jet
            
            std_pt = event.genJetPt[ijet]
            std_eta = event.genJetEta[ijet]
            
            if eventPass(std_pt, std_eta):  # making lab frame jet cuts
                dau_pts = event.genDau_pt[ijet]
                dau_etas = event.genDau_eta[ijet]
                dau_phis = event.genDau_phi[ijet]
                dau_charges = event.genDau_chg[ijet]

                jet_constituents = make_pseudojets(dau_pts, dau_etas, dau_phis, dau_charges) #check that make_pseudojets is optimised.

                # WTA Clustering
                wta_clustered = fastjet.ClusterSequence(jet_constituents, wta_def)
                wta_jets_tup = fastjet.sorted_by_pt(wta_clustered.inclusive_jets()) # is the sorting necessary? does this eat into compute time?
                
                if len(wta_jets_tup) > 0: # is this if statement necessary? will it ever be equal to zero?

                    wta_in_jet_frame, wta_mult = jetToJetFrame(wta_jets_tup[0]) # check that jetToJetFrame is optimised
                    b_name_wta = get_bin_key(wta_mult, analysis_bins)
                    
                    if b_name_wta: # i.e. it falls in one of the multiplicity bins (should be true always), this might be redundant.
                        hSig = bin_data[b_name_wta]['wta_hists']['sig']
                        hEPD = bin_data[b_name_wta]['wta_hists']['epd']
                        buildSignal(wta_in_jet_frame, hSig, hEPD)   # check that buildSignal is optimised and that it works with the right names, etc.
                        
                        pairs = len(wta_in_jet_frame) * (len(wta_in_jet_frame) - 1) // 2 # number of wta signal pairs in this jet
                        bin_data[b_name_wta]['wta_stats']['num_pairs'] += pairs # total number of pairs in the bin. used for background.
                        bin_data[b_name_wta]['wta_stats']['num_jets'] += 1      # number of jets in the bin. Used for bin's avg Nch per jet.
                        bin_data[b_name_wta]['wta_stats']['Nch'] += wta_mult    # total Nch for all jets in this bin. Avg Nch per jet is this divided by num_jets.

                # STD Clustering
                std_clustered = fastjet.ClusterSequence(jet_constituents, std_def)
                std_jets_tup = fastjet.sorted_by_pt(std_clustered.inclusive_jets()) # again is the sorting necessary?
                
                if len(std_jets_tup) > 0: #redundant or not?

                    std_in_jet_frame, std_mult = jetToJetFrame(std_jets_tup[0]) #check optimised
                    b_name_std = get_bin_key(std_mult, analysis_bins)
                    
                    if b_name_std:
                        hSig = bin_data[b_name_std]['std_hists']['sig']
                        hEPD = bin_data[b_name_std]['std_hists']['epd']
                        buildSignal(std_in_jet_frame, hSig, hEPD)           # check that names are fine
                        
                        pairs = len(std_in_jet_frame) * (len(std_in_jet_frame) - 1) // 2
                        bin_data[b_name_std]['std_stats']['num_pairs'] += pairs
                        bin_data[b_name_std]['std_stats']['num_jets'] += 1
                        bin_data[b_name_std]['std_stats']['Nch'] += std_mult

                # Critical C++ Memory Cleanup for FastJet Objects
                del wta_clustered   # don't need these cluster sequences anymore
                del std_clustered   
                
                #maybe delete std_jets_tup, std_in_jet_frame, anything else?
                
                gc.collect() #what exactly des this remove?

    # 4. Post-Processing: Loop over the populated dictionaries to build backgrounds and yields
    for b_name, data in bin_data.items():
        wta_stats = data['wta_stats']
        std_stats = data['std_stats']
        
        # Only process bins that actually caught jets
        if wta_stats['num_jets'] > 0 and std_stats['num_jets'] > 0:
            
            # Extract references
            hEPD_wta, hBkg_wta = data['wta_hists']['epd'], data['wta_hists']['bkg']
            hEPD_std, hBkg_std = data['std_hists']['epd'], data['std_hists']['bkg']
            
            # Build Backgrounds
            buildBkg(hEPD_wta, hBkg_wta, wta_stats['num_pairs'])
            buildBkg(hEPD_std, hBkg_std, std_stats['num_pairs'])


            #make sure epd and bkg are referencing different histogram objects every time the loop happens            

            # --- Yield processing and saveOutput() would be called here for this specific bin ---
            # Extract S/B ratios using your existing B(0,0) normalization logic

old stuff:

In [ ]:
#modified from Xiao's code
#clarify what the criteria is - he had something about 200 that i have not included

jetEtaCut = 1.6 #from PRL paper page 2
jetPtCut = 550  #from prl paper. This is the STANDARD pt, not wta pt.

def eventPass(jetPt, jetEta):  
    '''
    criteria for jet selection
    
    inputs:
        - jetEta: the jet's pseudorapidity, a positive or negative float
        - jetPt: the jet's momentum in GeV, a positive float             !!! USING STANDARD PT FOR CRITERIA, maybe later can try out with wta pt selection

    output: true if passed, false otherwise
    '''

    if(abs(jetEta)>jetEtaCut): return False
    if(jetPt < jetPtCut): return False

    return True



particlePtCut = 0.3 #from prl paper page 2
particleEtaCut = 2.4 #from prl paper page 2

def particlePass(particlePt, particleEta, particleCharge):
    '''
    criteria for particle selection

    inputs: 
        - particlePt: positive float for particle momentum magnitude
        - particleEta: float (positive or negative) for particle pseudorapidity
        - particleCharge: the particle's charge, not sure about the variable type

    output: true if passed, false otherwise
    '''
    if(particlePt < particlePtCut): return False
    if(abs(particleEta) > particleEtaCut): return False
    if (particleCharge is None or particleCharge==0): return False #only accepting charged particles

    return True



In [12]:
#from prelim file

def make_pseudojets(pts, etas, phis, charges):
    '''
    converts particles of one jet to a list of pseudojets. particle quality cuts are built in.
    inputs:
        - pts: a vector of particles for a given jet
        - etas: the corresponding pseudorapidities for each particle in the jet
        - phis: the corresponding azimuthal angles for each particle in the jet
        - charges: the charges of each particle in the jet
        
    output:
        - pj_list: a list of pseudojets, each pseudojet represents a particle in the jet
    '''
    
    pj_list = []

    for i in range(len(pts)): #looping through each particle in the jet
        
        if particlePass(pts[i], etas[i], charges[i]): #particle selection criteria
            p = fastjet.PseudoJet() #initialising a pseudojet
            p.reset_PtYPhiM(pts[i], etas[i], phis[i], 0.0) #defining the i-th particle as a pseudojet - is mass=0 fine?
            pj_list.append(p) #adding the particle to the list of pseudojets
    
    return pj_list #list of pseudojets


In [ ]:
#transforming values to jet frame

jt_lower = 0.3 #analysis can be redone with jt_lower = 0.5 as well.

def particleToJetFrame(jet_px, jet_py, jet_pz, particle):
    """
    Transforms one constituent particle to the jet frame. eta_star quality cuts reflected in 'include' output.
    
    Inputs:
        - jet_px, jet_py, jet_pz: The lab frame cartesian momentum 3-vector for the WHOLE jet
        - particle: A lab frame pseudoJet CONSTITUENT of that jet

    output:
        - jt: jet frame transverse momentum of the particle
        - eta_star: jet frame pseudorapidity of the particle
        - phi_star: jet frame phi of the particle
        - include: true if particle meets eta_star and jt criteria, false otherwise
    """

    # 1. Convert to ROOT TVector3
    # Use px, py, pz to ensure we have the full 3D vector
    p_jet = ROOT.TVector3(jet_px, jet_py, jet_pz)                 #momentum 3-vector of the whole jet
    p_part = ROOT.TVector3(particle.px(), particle.py(), particle.pz()) #momentum 3-vector of the particle

    # 1. Calculate Eta star (Relative Eta), discard non-qualifiers
    theta = p_part.Angle(p_jet)     #dot product angle between particle and jet momenta, range 0 to pi
    #cases for theta
    if theta == 0 or theta == math.pi: 
        return 0, 0, 0, False #on axis, not valid eta_star
    else: 
        eta_star = -math.log(math.tan(theta / 2.0)) #pseudorapidity formula for acceptable theta
        if abs(eta_star) > 5:
            return 0, 0, 0, False #include is false because absolute eta_star greater than 5 not allowed. 

        #what about eta_star less than 0.86? that is the anti-kt jet bdry. do i need to exclude stuff outside that?
        #eta_star values lower than 0.86 won't really happen because that is outside the theta = 0.8 anti-kt boundary.
        # so eta star is effectively limited to [0.86, 5]

    # 2. Calculate jT (Relative pT)
    jt = p_part.Perp(p_jet) #magnitude of part of particle momentum that is perpendicular to jet momentum
    if jt < jt_lower or jt > 3: #keeping 0.3geV<jt<3GeV . 
        return 0, 0, 0, False   #not soft so not plotted
    
    # 4. Calculate Phi star (Relative Phi)
    unit_jet = p_jet.Unit()
    z_axis = ROOT.TVector3(0, 0, 1)
    
    # Vector purely transverse to the jet axis
    v_pt = p_part - (unit_jet * p_part.Dot(unit_jet)) #magnitude of this is jt
    
    # Define the reference plane (jet axis & beam line)
    phi_origin = unit_jet.Cross(unit_jet.Cross(z_axis)) #phi = 0 vector, taken from Xiao's code
    phi_star = v_pt.Angle(phi_origin)
    
    # Determine the sign of phi_star
    if (phi_origin.Cross(v_pt.Unit())).Dot(unit_jet) < 0:
        phi_star = -phi_star

    return jt, eta_star, phi_star, True #if we reached this point without returning, then include should be true


def jetToJetFrame(labFrame_pj):
    '''
    transforms all constituents of a full jet from lab frame to the jet frame

    input: lab_pj, a jet pseudojet with coordinates measured in lab frame. The jet should already be clustered according to the desired scheme.

    outputs: 
        - ple_pj_list, a list of valid particle pseudojets with coordinates measured in the jet frame. (excludes hard jt)
        - jet_mult: the number of particles that meet in and out of jet criteria (doesn't exclude hard jt)
        note: len(ple_pj_list) != jet_mult because jet_mult includes hard particles and failed eta_star values
    '''
    #Extracting px, py, pz of the whole jet in lab frame
    jet_px =  labFrame_pj.px()
    jet_py = labFrame_pj.py()
    jet_pz = labFrame_pj.pz()
    
    ple_pj_list = [] #initialising list of particle pseudojets

    #looping through constituents
    jet_constituents = labFrame_pj.constituents()
    jet_mult = 0 #initialising multiplicity count
    for particle in jet_constituents:
        jet_mult+=1 # a valid clustered particle, but jt and eta_star not necessarily valid

        #finding jet frame coordinates for the particle
        jetFrame_pt, jetFrame_eta, jetFrame_phi, include = particleToJetFrame(jet_px, jet_py, jet_pz, particle)

        if include == True: #particle meets eta_star and jt criteria
            p_in_jet = fastjet.PseudoJet() #initialising a pseudojet     
            p_in_jet.reset_PtYPhiM(jetFrame_pt, jetFrame_eta, jetFrame_phi, 0.0) #redefining the particle in the jet frame (last index is mass)
            ple_pj_list.append(p_in_jet) #add the newly defined particle pseudojet to the list

        #maybe delete particle to free up memory?

    return ple_pj_list, jet_mult


In [25]:
#6 way histogram fill
def sixWayFill(hist, eta, phi, weight):
    '''
    helper function to do the 6-way symmetric histogram filling
    input: the 2d root histogram, the 2 inputs eta and phi, and a weight
    output: none, we are modifying the histogram
    '''
    hist.Fill(eta, phi, weight) #quadrant 1
    hist.Fill(-eta, phi, weight) #quadrant 2
    hist.Fill(-eta, -phi, weight) #quadrant 3
    hist.Fill(eta, -phi, weight) #quadrant 4
    hist.Fill(eta, 2*math.pi - phi, weight) #quadrant 1, phi wrap around pi
    hist.Fill(-eta, 2*math.pi - phi, weight) #quadrant 2, phi wrap around pi

In [ ]:
# do the 2 particle correlation

def buildSignal(pj_list, hSig, hEPD):
    '''
    fills in the signal and EPD histograms based on a jet represented by pj_list
    
    input: 
        - pj_list, a list of particle pseudojets for one jet. these should be in the jet frame, and in the right multiplicity bin
        - hSig, the signal histogram
        - hEPD, the event particle distribution histogram
    
    output:
        nothing. This just fills in the histograms.
    '''
    
    N_trig = len(pj_list) #JUST FOR THE LOOP, NOT THE HARD JT,FAILED ETA* INCLUSIVE MULTIPLICITY. that is jet_mult
    if N_trig < 2:
        return #no pairs can be made

    #build EPD
    for p in pj_list:
        corrected_phi = ROOT.TVector2.Phi_mpi_pi(p.phi()) #turning fastjet phi (0 to 2pi) into a -pi to pi range
        hEPD.Fill(p.eta(), corrected_phi, 1/N_trig) #filling in the trigger particle position in the EPD 


    #making the signal - is there a more efficient version of this than a nested for loop?
    for i in range(N_trig - 1):
        trig_eta = pj_list[i].eta()
        trig_phi = pj_list[i].phi()

        for j in range(i+1, N_trig): 
            track_eta = pj_list[j].eta()
            track_phi = pj_list[j].phi()
            delta_eta_star = abs(trig_eta - track_eta) #absolute dEta. Due to jet frame cuts, this is limited to 
            delta_phi_star = math.acos(math.cos(trig_phi - track_phi)) #limits dPhi to [0,pi]

            #weight = 1/Ntrig here, but it would involve the normalised energy product for an EEC
            #filling the signal symmetrically 6 ways
            sixWayFill(hSig, delta_eta_star, delta_phi_star, 1/N_trig) #weighting by 1/number of trigger particles in this jet
    
    #contribution of this one jet to signal and EPD are now filled
    #particles and pairs are weighted per trigger in the jet


In [ ]:
#build the background distribution once signal and EPD are filled
#method used here is the one described in the prl paper, not the one from github
def buildBkg(hEPD, hBkg_pre_corr, num_pairs):
    '''
    fills in the background histogram using the given EPD
    inputs:
        - hEPD, the signle-particle distribution
        - hBkg_pre_corr, the background histogram that needs filling, before it has been normalised by B(0,0)
        - num_pairs, the integer number of signal pairs.
    output:
        - fills and scales the background histogram so that it becomes hBkg, the corrected background histogram
    '''

    # 1. initialising c type coordinates
    eta1 = ctypes.c_double(0.0)
    phi1 = ctypes.c_double(0.0)
    eta2 = ctypes.c_double(0.0)
    phi2 = ctypes.c_double(0.0)

    # 2. loop to build background
    for _ in range(10*num_pairs): #background pairs are 10x the number of signal pairs, make sure it's an integer and not a float
        # Draw coords of two random particles from the EPD
        hEPD.GetRandom2(eta1, phi1)
        hEPD.GetRandom2(eta2, phi2)
        
        # 3. Extract the Python floats using .value
        e1 = eta1.value
        p1 = phi1.value
        e2 = eta2.value
        p2 = phi2.value
        
        # 4. Calculate kinematics using the extracted values
        delta_eta_star = abs(e1 - e2)
        delta_phi_star = math.acos(math.cos(p1 - p2))
        
        # 6-way fill for hBkg_pre_corr
        sixWayFill(hBkg_pre_corr, delta_eta_star, delta_phi_star, 1.0)

    #background is now filled the way described in the prl paper for the entire multiplicity bin
    #not yet corrected by B(0,0 though)

In [ ]:
# NOT BEING USED RN
# Doing signal / background to get yield. 
def makeYield(hSig, hBkg, num_jets):
    '''
    makes yield. run once for wta, once for std
    '''
    #find B(0,0)
    origin_bin = hBkg.FindBin(0.0, 0.0)     # 1. Find the global bin index that corresponds to x=0.0, y=0.0
    B_00 = hBkg.GetBinContent(origin_bin)   # 2. Extract the number from that bin (this is B(0,0) , unscaled by ntrig)

    #find yield
    hYield = hSig.Clone("hYield_wta")
    hYield.Divide(hBkg)      # This does S(dEta, dPhi) / B(dEta, dPhi)
    hYield.Scale(B_00)       # This multiplies every bin by the single number B(0,0) (unscaled, but hBkg is also unscaled so it cancels)
    hYield.Scale(1/num_jets)    #averaging over all jets

    return hYield



In [ ]:
# --- SAVING THE OUTPUT ---

def saveOutput(bin_name, hYield_wta, hSig_wta, hBkg_wta, hEPD_wta, Nassoc_wta, avg_Nch_wta, hYield_std, hSig_std, hBkg_std, hEPD_std, Nassoc_std, avg_Nch_std):
    '''
    saves the 2d histograms
    '''
    # 1. Define your specific output folder path
    #output_dir = "/Users/rohanjagadeesan/Desktop/Code/li_lab/WinnerTakeAll/output/2pc_output_histograms"
    output_dir = "/home/rj65/winner_take_all/output"    # for nots

    # Create the folder if it doesn't already exist
    os.makedirs(output_dir, exist_ok=True)

    # file name
    filename = f"Yield_Histograms_Mult_{bin_name}.root"

    # 3. Combine the folder path and the filename
    full_filepath = os.path.join(output_dir, filename)

    # 4. Open the ROOT file using the FULL path
    out_file = ROOT.TFile(full_filepath, "RECREATE")

    # 5. write the histograms - naming already done in initialisation
    hYield_wta.Write()
    hYield_std.Write()
    hSig_wta.Write()
    hSig_std.Write()
    hBkg_wta.Write()
    hBkg_std.Write()
    hEPD_wta.Write()
    hEPD_std.Write()

    # 6. Create and Write the TParameters for Nassoc, and avg Nch per jet
    param_nassoc_wta = ROOT.TParameter('double')("Nassoc_WTA", Nassoc_wta)
    param_nassoc_std = ROOT.TParameter('double')("Nassoc_STD", Nassoc_std)
    param_nassoc_wta.Write()
    param_nassoc_std.Write()

    param_avg_Nch_wta = ROOT.TParameter('double')("avg_Nch_WTA", avg_Nch_wta)
    param_avg_Nch_std = ROOT.TParameter('double')("avg_Nch_STD", avg_Nch_std)
    param_avg_Nch_wta.Write()
    param_avg_Nch_std.Write()

    out_file.Close()

    print(f"Successfully saved yields to {full_filepath}")

### Cell below runs all the analysis, and saves the output

In [ ]:
#doing the thing for all the bins

analysis_bins = [ [0,25], [25,36], [36,48], [48,60], [60,71], [71,78], [78,91], [91,97], [97,1000] ]


for mult_bin in analysis_bins:
    runAnalysis(tree, mult_bin)

ROOT.gDirectory.Clear()

Successfully saved yields to /Users/rohanjagadeesan/Desktop/Code/li_lab/WinnerTakeAll/output/2pc_output_histograms/Yield_Histograms_Mult_0_25.root
Successfully saved yields to /Users/rohanjagadeesan/Desktop/Code/li_lab/WinnerTakeAll/output/2pc_output_histograms/Yield_Histograms_Mult_25_36.root
Successfully saved yields to /Users/rohanjagadeesan/Desktop/Code/li_lab/WinnerTakeAll/output/2pc_output_histograms/Yield_Histograms_Mult_36_48.root
Successfully saved yields to /Users/rohanjagadeesan/Desktop/Code/li_lab/WinnerTakeAll/output/2pc_output_histograms/Yield_Histograms_Mult_48_60.root
Successfully saved yields to /Users/rohanjagadeesan/Desktop/Code/li_lab/WinnerTakeAll/output/2pc_output_histograms/Yield_Histograms_Mult_60_71.root
Successfully saved yields to /Users/rohanjagadeesan/Desktop/Code/li_lab/WinnerTakeAll/output/2pc_output_histograms/Yield_Histograms_Mult_71_78.root
Successfully saved yields to /Users/rohanjagadeesan/Desktop/Code/li_lab/WinnerTakeAll/output/2pc_output_histogram